In [ ]:
# @title 1. Setup, TPU, Drive, and config

!pip install -q flax optax tensorflow tensorflow-datasets scipy pandas scikit-learn

import os
import re
import gc
import math
import itertools
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
import scipy.stats as stats

import jax
import jax.numpy as jnp
from jax import random
import flax.linen as nn
from flax import serialization

import tensorflow as tf
import tensorflow_datasets as tfds

from tqdm.notebook import tqdm
from google.colab import drive

# TPU setup
try:
    import jax.tools.colab_tpu
    jax.tools.colab_tpu.setup_tpu()
except Exception:
    pass

print("JAX backend:", jax.default_backend())
print("Local device count:", jax.local_device_count())

# -------------------------
# User options
# -------------------------
# "random", "pca", or "both"
BASIS_MODE = "both"

# If you really want exact full-dataset Procrustes, flip this to True.
# It is much slower than the subset version.
RUN_PROCRUSTES_FULL = False

# -------------------------
# Mount Drive
# -------------------------
drive.mount("/content/drive", force_remount=True)

# -------------------------
# Paths
# -------------------------
MODEL_SAVE_PATH = Path("/content/drive/MyDrive/jax_models")

RUN_TAG = f"tiny10_layer_table_fast_{BASIS_MODE}"
LOCAL_WORKDIR = Path("/content/tiny10_layer_table_fast")
DRIVE_OUTDIR = Path("/content/drive/MyDrive/layer_matching_table_data") / RUN_TAG

LOCAL_WORKDIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTDIR.mkdir(parents=True, exist_ok=True)

print("Model path:", MODEL_SAVE_PATH)
print("Local workdir:", LOCAL_WORKDIR)
print("Drive outdir:", DRIVE_OUTDIR)

# -------------------------
# Global config
# -------------------------
N_FULL = 2000
N_POINTWISE = 256

K_TABLE = [16, 32, 64, 128]
K_POINTWISE_MAX = max(K_TABLE)
K_REF = 768
EPS_REG = 1e-4
RANDOM_PARENT_SEED = 0

NUM_NETWORKS = 10

# Chunking tuned for v6e-1 / high-RAM Colab
CKA_CHUNK = 64
ACT_CACHE_CHUNK = 64
FULL_GEOM_CHUNK = 64
POINTWISE_GEOM_CHUNK = 32

def shard_for_pmap(x):
    n = x.shape[0]
    num_devices = jax.local_device_count()
    n_per_device = n // num_devices
    n_trunc = n_per_device * num_devices
    if n_trunc < n:
        x = x[:n_trunc]
    return x.reshape((num_devices, n_per_device) + x.shape[1:])

print("Config ready.")

In [ ]:
# @title 2. Model definition, model loading, and data loading

class Tiny10(nn.Module):
    num_classes: int = 10

    @nn.compact
    def __call__(self, x, train: bool):
        activations = {}
        norm = partial(nn.BatchNorm, use_running_average=not train, momentum=0.9)

        x = nn.Conv(features=16, kernel_size=(3, 3), padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv1_relu"] = x

        x = nn.Conv(features=16, kernel_size=(3, 3), padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv2_relu"] = x

        x = nn.Conv(features=32, kernel_size=(3, 3), strides=2, padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv3_relu"] = x

        x = nn.Conv(features=32, kernel_size=(3, 3), padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv4_relu"] = x

        x = nn.Conv(features=32, kernel_size=(3, 3), padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv5_relu"] = x

        x = nn.Conv(features=64, kernel_size=(3, 3), strides=2, padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv6_relu"] = x

        x = nn.Conv(features=64, kernel_size=(3, 3), padding=0, use_bias=False)(x)
        x = norm()(x); x = nn.relu(x); activations["conv7_relu"] = x

        x = nn.Conv(features=64, kernel_size=(1, 1), padding=0)(x)
        x = norm()(x); x = nn.relu(x); activations["conv8_relu"] = x

        x = jnp.mean(x, axis=(1, 2))
        x = nn.Dense(features=self.num_classes)(x)
        return x, activations

def natural_key(s):
    return [int(c) if c.isdigit() else c for c in re.split(r"(\d+)", s)]

key = jax.random.PRNGKey(0)
dummy_input = jnp.ones((1, 32, 32, 3))
init_vars = Tiny10().init(key, dummy_input, train=False)

_, dummy_acts = Tiny10().apply(init_vars, dummy_input, train=False)
LAYER_NAMES = sorted(list(dummy_acts.keys()), key=natural_key)
print("Layers:", LAYER_NAMES)

# Load models
loaded_models = []
for i in range(NUM_NETWORKS):
    path = MODEL_SAVE_PATH / f"tiny10_model_{i}.msgpack"
    if not path.exists():
        raise FileNotFoundError(path)
    with open(path, "rb") as f:
        data = serialization.from_bytes(init_vars, f.read())
    loaded_models.append(data)

print(f"Loaded {len(loaded_models)} Tiny10 checkpoints.")

def get_data_numpy(count):
    ds = tfds.load("cifar10", split="train", shuffle_files=True)
    ds = ds.map(lambda x: (tf.cast(x["image"], tf.float32) - 127.5) / 127.5)
    ds = ds.batch(count).take(1)
    for img in ds:
        return img.numpy()

print(f"Loading {N_FULL} full images and {N_POINTWISE} comparable pointwise images...")
X_full = get_data_numpy(N_FULL)
X_pointwise = X_full[:N_POINTWISE]

X_full_sharded = shard_for_pmap(X_full)
X_pointwise_sharded = shard_for_pmap(X_pointwise)

print("X_full:", X_full.shape, "->", X_full_sharded.shape)
print("X_pointwise:", X_pointwise.shape, "->", X_pointwise_sharded.shape)

In [ ]:
# @title 3. Activation helpers, geometry engine, and metric functions

# -------------------------
# A. Activation extraction
# -------------------------

@partial(jax.pmap, in_axes=(None, None, 0), static_broadcasted_argnums=(1,))
def compute_act_chunk(variables, layer_name, x_batch):
    _, acts = Tiny10().apply(variables, x_batch, train=False)
    act = acts[layer_name]
    return act.reshape(act.shape[0], -1)

def get_activation_matrix_cpu_chunked(variables, layer_name, x_sharded, chunk_size=64):
    n_per_device = x_sharded.shape[1]
    out = []
    for i in range(0, n_per_device, chunk_size):
        x_chunk = x_sharded[:, i:i + chunk_size]
        acts_chunk = compute_act_chunk(variables, layer_name, x_chunk)
        acts_chunk = np.array(acts_chunk).reshape(-1, acts_chunk.shape[-1]).astype(np.float32)
        out.append(acts_chunk)
    return np.concatenate(out, axis=0)

def center_rows(X):
    return X - X.mean(axis=0, keepdims=True)

def get_cka_gram_from_centered_acts(Xc):
    return (Xc @ Xc.T).astype(np.float32)

def compute_cka_from_cache(entry_x, entry_y):
    Xc = entry_x["Xc"]
    Yc = entry_y["Xc"]
    K = get_cka_gram_from_centered_acts(Xc)
    L = get_cka_gram_from_centered_acts(Yc)
    num = np.trace(K @ L)
    den = np.linalg.norm(K) * np.linalg.norm(L) + 1e-12
    return float(num / den)

def compute_left_svd_factors_from_centered_acts(Xc, floor=1e-12):
    """
    Returns thin left singular factors from the sample-side Gram XX^T.
    Works even when feature dimension differs across layers.
    """
    G = (Xc @ Xc.T).astype(np.float64)   # (n, n)
    evals, U = np.linalg.eigh(0.5 * (G + G.T))
    keep = evals > floor
    evals = evals[keep]
    U = U[:, keep]
    s = np.sqrt(np.maximum(evals, floor))
    order = np.argsort(s)[::-1]
    s = s[order].astype(np.float32)
    U = U[:, order].astype(np.float32)
    return U, s

def compute_procrustes_similarity_from_cache(entry_x, entry_y):
    """
    Exact orthogonal Procrustes similarity:
        ||X^T Y||_* / (||X||_F ||Y||_F)

    We compute it through thin sample-side SVD factors:
        X = Ux Sx Vx^T,  Y = Uy Sy Vy^T
    so
        ||X^T Y||_* = || Sx (Ux^T Uy) Sy ||_*
    and this only involves at most n x n matrices.
    """
    Ux, sx = entry_x["U"], entry_x["s"]
    Uy, sy = entry_y["U"], entry_y["s"]

    core = (sx[:, None] * (Ux.T @ Uy).astype(np.float32)) * sy[None, :]
    nuc = np.linalg.svd(core, compute_uv=False, full_matrices=False).sum()

    denom = (np.linalg.norm(sx) * np.linalg.norm(sy)) + 1e-12
    return float(nuc / denom)

# -------------------------
# B. Geometry engine
# -------------------------

class GeometryEngine:
    def __init__(self, model_class, input_shape=(32, 32, 3)):
        self.model = model_class()
        self.flat_dim = int(np.prod(input_shape))

    def generate_projections(self, k, seed):
        key = random.PRNGKey(seed)
        P = random.normal(key, (self.flat_dim, k))
        P, _ = jnp.linalg.qr(P)
        return P

    @staticmethod
    @partial(jax.pmap, in_axes=(None, None, None, 0, 0), static_broadcasted_argnums=(0, 1))
    def compute_gram_chunk(apply_fn, layer_name, vars, x_batch, P_device):
        def target_fn(x):
            _, acts = apply_fn(vars, x[None, ...], train=False)
            return acts[layer_name].reshape(-1)

        def jvp_fn(img):
            tangents = P_device.T.reshape((-1,) + img.shape)

            def single_jvp(t):
                _, tangent_out = jax.jvp(target_fn, (img,), (t,))
                return tangent_out

            return jax.vmap(single_jvp)(tangents)

        batch_jvps = jax.vmap(jvp_fn)(x_batch)
        grams = jnp.matmul(batch_jvps, jnp.swapaxes(batch_jvps, 1, 2))
        return jnp.sum(grams, axis=0)

    @staticmethod
    @partial(jax.pmap, in_axes=(None, None, None, 0, 0), static_broadcasted_argnums=(0, 1))
    def compute_gram_chunk_per_sample(apply_fn, layer_name, vars, x_batch, P_device):
        def target_fn(x):
            _, acts = apply_fn(vars, x[None, ...], train=False)
            return acts[layer_name].reshape(-1)

        def jvp_fn(img):
            tangents = P_device.T.reshape((-1,) + img.shape)

            def single_jvp(t):
                _, tangent_out = jax.jvp(target_fn, (img,), (t,))
                return tangent_out

            return jax.vmap(single_jvp)(tangents)

        batch_jvps = jax.vmap(jvp_fn)(x_batch)
        grams = jnp.matmul(batch_jvps, jnp.swapaxes(batch_jvps, 1, 2))
        return grams

    def get_gram(self, vars, layer, data_sharded, k, chunk_size=64, P_override=None):
        P = jnp.asarray(P_override)
        P_repl = jax.device_put_replicated(P, jax.local_devices())

        n_per_device = data_sharded.shape[1]
        total = np.zeros((k, k), dtype=np.float32)

        for i in range(0, n_per_device, chunk_size):
            x_chunk = data_sharded[:, i:i + chunk_size]
            grams_chunk_sum = self.compute_gram_chunk(
                self.model.apply, layer, vars, x_chunk, P_repl
            )
            total += np.sum(np.array(grams_chunk_sum), axis=0).astype(np.float32)
            del grams_chunk_sum
            gc.collect()

        total_n = data_sharded.shape[0] * data_sharded.shape[1]
        return total / total_n

    def get_grams_per_sample(self, vars, layer, data_sharded, k, chunk_size=32, P_override=None):
        P = jnp.asarray(P_override)
        P_repl = jax.device_put_replicated(P, jax.local_devices())

        n_per_device = data_sharded.shape[1]
        out = []

        for i in range(0, n_per_device, chunk_size):
            x_chunk = data_sharded[:, i:i + chunk_size]
            grams_chunk = self.compute_gram_chunk_per_sample(
                self.model.apply, layer, vars, x_chunk, P_repl
            )
            grams_chunk = np.array(grams_chunk).reshape(-1, k, k).astype(np.float32)
            out.append(grams_chunk)
            del grams_chunk
            gc.collect()

        return np.concatenate(out, axis=0)

geo_engine = GeometryEngine(Tiny10)

# -------------------------
# C. SPD helpers
# -------------------------

def symmetrize(A):
    return 0.5 * (A + A.T)

def symmetrize_batch(A):
    return 0.5 * (A + np.swapaxes(A, -1, -2))

def normalize_trace_single(G, eps=1e-12):
    G = symmetrize(np.asarray(G, dtype=np.float32))
    tr = float(np.trace(G))
    return G / max(tr, eps)

def normalize_trace_batch(Gs, eps=1e-12):
    Gs = symmetrize_batch(np.asarray(Gs, dtype=np.float32))
    tr = np.trace(Gs, axis1=-2, axis2=-1)[..., None, None]
    return Gs / np.maximum(tr, eps)

def spd_lift_trace_scaled(G, eps_reg=1e-4):
    G = symmetrize(np.asarray(G, dtype=np.float32))
    k = G.shape[0]
    tr = float(np.trace(G))
    scale = tr / k if tr > 0 else 1.0
    return G + (eps_reg * scale) * np.eye(k, dtype=np.float32)

def spd_lift_trace_scaled_batch(Gs, eps_reg=1e-4):
    Gs = symmetrize_batch(np.asarray(Gs, dtype=np.float32))
    k = Gs.shape[-1]
    tr = np.trace(Gs, axis1=-2, axis2=-1)[..., None, None]
    scale = np.where(tr > 0, tr / k, 1.0)
    I = np.eye(k, dtype=np.float32)[None, :, :]
    return Gs + (eps_reg * scale) * I

def matrix_inv_sqrt_spd_batch(A, floor=1e-12):
    A = symmetrize_batch(A)
    evals, evecs = np.linalg.eigh(A)
    inv_sqrt = 1.0 / np.sqrt(np.maximum(evals, floor))
    return evecs @ (inv_sqrt[..., None, :] * np.swapaxes(evecs, -1, -2))

def compute_airm_spd(A, B, floor=1e-12):
    A = symmetrize(np.asarray(A, dtype=np.float64))
    B = symmetrize(np.asarray(B, dtype=np.float64))
    evals, evecs = np.linalg.eigh(A)
    A_inv_sqrt = evecs @ np.diag(1.0 / np.sqrt(np.maximum(evals, floor))) @ evecs.T
    mid = symmetrize(A_inv_sqrt @ B @ A_inv_sqrt)
    evals_mid = np.linalg.eigvalsh(mid)
    return float(np.sqrt(np.sum(np.log(np.maximum(evals_mid, floor)) ** 2)))

def compute_airm_spd_batch(A, B, floor=1e-12):
    A_inv_sqrt = matrix_inv_sqrt_spd_batch(A, floor=floor)
    mid = symmetrize_batch(A_inv_sqrt @ B @ A_inv_sqrt)
    evals_mid = np.linalg.eigvalsh(mid)
    return np.sqrt(np.sum(np.log(np.maximum(evals_mid, floor)) ** 2, axis=-1))

def compute_sr_spd_batch(A, B, floor=1e-12):
    A_inv_sqrt = matrix_inv_sqrt_spd_batch(A, floor=floor)
    mid = symmetrize_batch(A_inv_sqrt @ B @ A_inv_sqrt)
    lam = np.linalg.eigvalsh(mid)
    lam_min = np.maximum(lam[..., 0], floor)
    lam_max = np.maximum(lam[..., -1], floor)
    return 1.0 - np.sqrt(lam_min / lam_max)

def compute_sras_distance_from_grams(G1, G2, eps_reg=EPS_REG, shape_only=False):
    G1 = np.asarray(G1, dtype=np.float32)
    G2 = np.asarray(G2, dtype=np.float32)

    if shape_only:
        G1 = normalize_trace_single(G1)
        G2 = normalize_trace_single(G2)

    k = G1.shape[0]
    G1_tilde = spd_lift_trace_scaled(G1, eps_reg=eps_reg)
    G2_tilde = spd_lift_trace_scaled(G2, eps_reg=eps_reg)
    return compute_airm_spd(G1_tilde, G2_tilde) / np.sqrt(k)

def compute_pw_airm_distance_from_per_sample_grams(Gs1, Gs2, eps_reg=EPS_REG):
    Gs1 = np.asarray(Gs1, dtype=np.float32)
    Gs2 = np.asarray(Gs2, dtype=np.float32)
    k = Gs1.shape[-1]
    A = spd_lift_trace_scaled_batch(Gs1, eps_reg=eps_reg)
    B = spd_lift_trace_scaled_batch(Gs2, eps_reg=eps_reg)
    d = compute_airm_spd_batch(A, B) / np.sqrt(k)
    return float(np.mean(d))

def compute_pw_msa_distance_from_per_sample_grams(Gs1, Gs2, eps_reg=EPS_REG):
    Gs1 = np.asarray(Gs1, dtype=np.float32)
    Gs2 = np.asarray(Gs2, dtype=np.float32)
    A = spd_lift_trace_scaled_batch(Gs1, eps_reg=eps_reg)
    B = spd_lift_trace_scaled_batch(Gs2, eps_reg=eps_reg)
    d = compute_sr_spd_batch(A, B)
    return float(np.mean(d))

# -------------------------
# D. Pair utilities
# -------------------------

PAIR_LIST = list(itertools.combinations(range(len(loaded_models)), 2))

def pair_accuracy_from_sim_map(sim_map):
    n = sim_map.shape[0]
    hits = 0
    for i in range(n):
        hits += int(np.argmax(sim_map[i, :]) == i)
    for j in range(n):
        hits += int(np.argmax(sim_map[:, j]) == j)
    return 100.0 * hits / (2 * n)

def sem_or_zero(x):
    x = np.asarray(x, dtype=np.float64)
    if len(x) <= 1:
        return 0.0
    return float(stats.sem(x))

def make_summary_row(method, family, regime, K, n_images, pair_accs, comparison_cost, note=""):
    pair_accs = np.asarray(pair_accs, dtype=np.float64)
    return {
        "method": method,
        "family": family,
        "regime": regime,
        "K": ("" if K is None else int(K)),
        "n_images": int(n_images),
        "n_pairs": int(len(pair_accs)),
        "pair_accuracy_mean_pct": float(pair_accs.mean()),
        "pair_accuracy_sem_pct": sem_or_zero(pair_accs),
        "overall_accuracy_pct": float(pair_accs.mean()),
        "comparison_cost": comparison_cost,
        "note": note,
    }

def make_pair_rows(method, family, regime, K, pair_accs):
    rows = []
    for (m1, m2), acc in zip(PAIR_LIST, pair_accs):
        rows.append({
            "method": method,
            "family": family,
            "regime": regime,
            "K": ("" if K is None else int(K)),
            "model_i": int(m1),
            "model_j": int(m2),
            "pair_accuracy_pct": float(acc),
        })
    return rows

def evaluate_metric_pair_accs(item_lookup, metric_fn, is_dist=False, desc=None):
    n_layers = len(LAYER_NAMES)
    pair_accs = []
    iterator = tqdm(PAIR_LIST, desc=desc, leave=False) if desc is not None else PAIR_LIST

    for idx1, idx2 in iterator:
        pair_map = np.empty((n_layers, n_layers), dtype=np.float32)
        for i, l1 in enumerate(LAYER_NAMES):
            x1 = item_lookup[idx1][l1]
            for j, l2 in enumerate(LAYER_NAMES):
                x2 = item_lookup[idx2][l2]
                pair_map[i, j] = metric_fn(x1, x2)
        sim_map = np.exp(-pair_map).astype(np.float32) if is_dist else pair_map.astype(np.float32)
        pair_accs.append(pair_accuracy_from_sim_map(sim_map))

    return np.array(pair_accs, dtype=np.float32)

def evaluate_multi_k_geometry_pair_accs(prefix_mean_lookup, prefix_ps_lookup, k_list, desc_prefix=""):
    """
    Faster multi-K evaluation in one pass over model-pairs and layer-pairs.
    """
    n_layers = len(LAYER_NAMES)
    pair_maps = {
        "sras_full": {k: [] for k in k_list},
        "sras_shape": {k: [] for k in k_list},
        "pwairm": {k: [] for k in k_list},
        "msa": {k: [] for k in k_list},
    }

    iterator = tqdm(PAIR_LIST, desc=desc_prefix, leave=False)

    for idx1, idx2 in iterator:
        pair_map_sras_full = {k: np.empty((n_layers, n_layers), dtype=np.float32) for k in k_list}
        pair_map_sras_shape = {k: np.empty((n_layers, n_layers), dtype=np.float32) for k in k_list}
        pair_map_pwairm = {k: np.empty((n_layers, n_layers), dtype=np.float32) for k in k_list}
        pair_map_msa = {k: np.empty((n_layers, n_layers), dtype=np.float32) for k in k_list}

        for i, l1 in enumerate(LAYER_NAMES):
            mean1 = prefix_mean_lookup[idx1][l1]
            ps1 = prefix_ps_lookup[idx1][l1]
            for j, l2 in enumerate(LAYER_NAMES):
                mean2 = prefix_mean_lookup[idx2][l2]
                ps2 = prefix_ps_lookup[idx2][l2]

                for k in k_list:
                    g1 = mean1[:k, :k]
                    g2 = mean2[:k, :k]
                    p1 = ps1[:, :k, :k]
                    p2 = ps2[:, :k, :k]

                    pair_map_sras_full[k][i, j] = compute_sras_distance_from_grams(g1, g2, eps_reg=EPS_REG, shape_only=False)
                    pair_map_sras_shape[k][i, j] = compute_sras_distance_from_grams(g1, g2, eps_reg=EPS_REG, shape_only=True)
                    pair_map_pwairm[k][i, j] = compute_pw_airm_distance_from_per_sample_grams(p1, p2, eps_reg=EPS_REG)
                    pair_map_msa[k][i, j] = compute_pw_msa_distance_from_per_sample_grams(p1, p2, eps_reg=EPS_REG)

        for k in k_list:
            pair_maps["sras_full"][k].append(pair_accuracy_from_sim_map(np.exp(-pair_map_sras_full[k])))
            pair_maps["sras_shape"][k].append(pair_accuracy_from_sim_map(np.exp(-pair_map_sras_shape[k])))
            pair_maps["pwairm"][k].append(pair_accuracy_from_sim_map(np.exp(-pair_map_pwairm[k])))
            pair_maps["msa"][k].append(pair_accuracy_from_sim_map(np.exp(-pair_map_msa[k])))

    return {
        metric: {k: np.array(accs, dtype=np.float32) for k, accs in metric_dict.items()}
        for metric, metric_dict in pair_maps.items()
    }

print("Helpers ready.")

In [ ]:
# @title 4. Cache activations and compute activation baselines (linear CKA + Procrustes)

summary_rows = []
pair_rows = []

def build_activation_cache(x_sharded, desc, include_procrustes=False):
    cache = {}
    for i in tqdm(range(len(loaded_models)), desc=desc):
        cache[i] = {}
        for layer in LAYER_NAMES:
            X = get_activation_matrix_cpu_chunked(
                loaded_models[i],
                layer,
                x_sharded,
                chunk_size=ACT_CACHE_CHUNK,
            )
            Xc = center_rows(X).astype(np.float32)

            entry = {"Xc": Xc}
            if include_procrustes:
                U, s = compute_left_svd_factors_from_centered_acts(Xc)
                entry["U"] = U
                entry["s"] = s

            cache[i][layer] = entry
        gc.collect()
    return cache

# Full activation cache for exact linear CKA
act_cache_full = build_activation_cache(
    X_full_sharded,
    desc="Activation cache full (2000)",
    include_procrustes=RUN_PROCRUSTES_FULL,
)

cka_full_pair_accs = evaluate_metric_pair_accs(
    act_cache_full,
    metric_fn=compute_cka_from_cache,
    is_dist=False,
    desc="Linear CKA full",
)
summary_rows.append(
    make_summary_row(
        method="cka_linear",
        family="activation",
        regime="full_2000",
        K=None,
        n_images=N_FULL,
        pair_accs=cka_full_pair_accs,
        comparison_cost="activation Gram",
        note="Full Tiny10 benchmark",
    )
)
pair_rows.extend(make_pair_rows("cka_linear", "activation", "full_2000", None, cka_full_pair_accs))

# Comparable subset activation cache for exact linear CKA + Procrustes
act_cache_subset = build_activation_cache(
    X_pointwise_sharded,
    desc="Activation cache subset (256)",
    include_procrustes=True,
)

cka_subset_pair_accs = evaluate_metric_pair_accs(
    act_cache_subset,
    metric_fn=compute_cka_from_cache,
    is_dist=False,
    desc="Linear CKA subset",
)
summary_rows.append(
    make_summary_row(
        method="cka_linear",
        family="activation",
        regime="subset_256",
        K=None,
        n_images=N_POINTWISE,
        pair_accs=cka_subset_pair_accs,
        comparison_cost="activation Gram",
        note="Comparable subset used for local baselines",
    )
)
pair_rows.extend(make_pair_rows("cka_linear", "activation", "subset_256", None, cka_subset_pair_accs))

procrustes_subset_pair_accs = evaluate_metric_pair_accs(
    act_cache_subset,
    metric_fn=compute_procrustes_similarity_from_cache,
    is_dist=False,
    desc="Procrustes subset",
)
summary_rows.append(
    make_summary_row(
        method="procrustes",
        family="activation",
        regime="subset_256",
        K=None,
        n_images=N_POINTWISE,
        pair_accs=procrustes_subset_pair_accs,
        comparison_cost="subset sample-side SVD",
        note="Exact orthogonal Procrustes on comparable subset",
    )
)
pair_rows.extend(make_pair_rows("procrustes", "activation", "subset_256", None, procrustes_subset_pair_accs))

if RUN_PROCRUSTES_FULL:
    procrustes_full_pair_accs = evaluate_metric_pair_accs(
        act_cache_full,
        metric_fn=compute_procrustes_similarity_from_cache,
        is_dist=False,
        desc="Procrustes full (slow)",
    )
    summary_rows.append(
        make_summary_row(
            method="procrustes",
            family="activation",
            regime="full_2000",
            K=None,
            n_images=N_FULL,
            pair_accs=procrustes_full_pair_accs,
            comparison_cost="full sample-side SVD",
            note="Exact orthogonal Procrustes on full 2000-image set",
        )
    )
    pair_rows.extend(make_pair_rows("procrustes", "activation", "full_2000", None, procrustes_full_pair_accs))

activation_summary_df = pd.DataFrame(summary_rows)
display(activation_summary_df.sort_values(["method", "regime"]))

In [ ]:
# @title 5. Build selected task-family parents

def make_random_parent(flat_dim, k, seed):
    key = random.PRNGKey(seed)
    P = random.normal(key, (flat_dim, k))
    P, _ = jnp.linalg.qr(P)
    return np.array(P, dtype=np.float32)

flat_dim = int(np.prod(X_full.shape[1:]))

families = {}

if BASIS_MODE in ["random", "both"]:
    P_RANDOM_REF = make_random_parent(flat_dim, K_REF, RANDOM_PARENT_SEED)
    P_RANDOM_POINTWISE = P_RANDOM_REF[:, :K_POINTWISE_MAX]
    families["random"] = {
        "P_ref": P_RANDOM_REF,
        "P_pointwise": P_RANDOM_POINTWISE,
    }
    print("Random parent built:", P_RANDOM_REF.shape)

if BASIS_MODE in ["pca", "both"]:
    X_flat = X_full.reshape(X_full.shape[0], -1).astype(np.float32)
    X_flat_centered = X_flat - X_flat.mean(axis=0, keepdims=True)

    U, S, Vt = np.linalg.svd(X_flat_centered, full_matrices=False)
    if K_REF > Vt.shape[0]:
        raise ValueError(f"K_REF={K_REF} exceeds PCA rank {Vt.shape[0]}")

    P_PCA_REF = Vt[:K_REF].T.astype(np.float32)
    P_PCA_POINTWISE = P_PCA_REF[:, :K_POINTWISE_MAX]

    explained_var = (S ** 2) / (X_flat_centered.shape[0] - 1)
    explained_ratio = explained_var / explained_var.sum()

    pca_meta = pd.DataFrame({
        "pc_index_1_based": np.arange(1, len(S) + 1),
        "singular_value": S,
        "explained_variance": explained_var,
        "explained_variance_ratio": explained_ratio,
        "cumulative_explained_variance_ratio": np.cumsum(explained_ratio),
    })
    pca_meta.to_csv(DRIVE_OUTDIR / "pca_family_metadata.csv", index=False)

    families["whitened_pca"] = {
        "P_ref": P_PCA_REF,
        "P_pointwise": P_PCA_POINTWISE,
    }
    print("PCA parent built:", P_PCA_REF.shape)
    for k in K_TABLE:
        print(f"PCA cumulative variance at K={k}: {np.cumsum(explained_ratio)[k-1]:.4f}")

print("Families selected:", list(families.keys()))

In [ ]:
# @title 6. Compute family-dependent rows: S-RAS full/shape, pwAIRM, MSA

def compute_family_rows(family_name, P_ref, P_pointwise):
    fam_summary_rows = []
    fam_pair_rows = []

    print(f"\n===== FAMILY: {family_name} =====")

    # ---------------------------------
    # A. Full 2000-image reference at K=768
    # ---------------------------------
    print(f"[{family_name}] Computing full-dataset averaged grams at K={K_REF}...")
    full_ref_lookup = {}
    for i in tqdm(range(len(loaded_models)), desc=f"{family_name} full mean grams"):
        full_ref_lookup[i] = {}
        for layer in LAYER_NAMES:
            full_ref_lookup[i][layer] = geo_engine.get_gram(
                loaded_models[i],
                layer,
                X_full_sharded,
                k=K_REF,
                chunk_size=FULL_GEOM_CHUNK,
                P_override=P_ref,
            )
        gc.collect()

    sras_ref_full_pair_accs = evaluate_metric_pair_accs(
        full_ref_lookup,
        lambda g1, g2: compute_sras_distance_from_grams(g1, g2, eps_reg=EPS_REG, shape_only=False),
        is_dist=True,
        desc=f"{family_name} S-RAS full K={K_REF}",
    )
    fam_summary_rows.append(
        make_summary_row(
            method="sras_full",
            family=family_name,
            regime="full_ref_2000",
            K=K_REF,
            n_images=N_FULL,
            pair_accs=sras_ref_full_pair_accs,
            comparison_cost="dataset-level O(K^3)",
            note="Higher-K reference on full 2000-image set",
        )
    )
    fam_pair_rows.extend(make_pair_rows("sras_full", family_name, "full_ref_2000", K_REF, sras_ref_full_pair_accs))

    sras_ref_shape_pair_accs = evaluate_metric_pair_accs(
        full_ref_lookup,
        lambda g1, g2: compute_sras_distance_from_grams(g1, g2, eps_reg=EPS_REG, shape_only=True),
        is_dist=True,
        desc=f"{family_name} S-RAS shape K={K_REF}",
    )
    fam_summary_rows.append(
        make_summary_row(
            method="sras_shape",
            family=family_name,
            regime="full_ref_2000",
            K=K_REF,
            n_images=N_FULL,
            pair_accs=sras_ref_shape_pair_accs,
            comparison_cost="dataset-level O(K^3)",
            note="Higher-K reference on full 2000-image set",
        )
    )
    fam_pair_rows.extend(make_pair_rows("sras_shape", family_name, "full_ref_2000", K_REF, sras_ref_shape_pair_accs))

    del full_ref_lookup
    gc.collect()

    # ---------------------------------
    # B. Comparable 256-image subset per-sample grams at Kmax=128
    # ---------------------------------
    print(f"[{family_name}] Computing subset per-sample grams at K={K_POINTWISE_MAX}...")
    subset_ps_lookup = {}
    subset_mean_lookup = {}
    for i in tqdm(range(len(loaded_models)), desc=f"{family_name} pointwise grams"):
        subset_ps_lookup[i] = {}
        subset_mean_lookup[i] = {}
        for layer in LAYER_NAMES:
            grams_ps = geo_engine.get_grams_per_sample(
                loaded_models[i],
                layer,
                X_pointwise_sharded,
                k=K_POINTWISE_MAX,
                chunk_size=POINTWISE_GEOM_CHUNK,
                P_override=P_pointwise,
            )
            subset_ps_lookup[i][layer] = grams_ps
            subset_mean_lookup[i][layer] = grams_ps.mean(axis=0).astype(np.float32)
        gc.collect()

    # ---------------------------------
    # C. Multi-K subset evaluation in one pass
    # ---------------------------------
    multi = evaluate_multi_k_geometry_pair_accs(
        prefix_mean_lookup=subset_mean_lookup,
        prefix_ps_lookup=subset_ps_lookup,
        k_list=K_TABLE,
        desc_prefix=f"{family_name} multi-K local baselines",
    )

    for method in ["sras_full", "sras_shape", "pwairm", "msa"]:
        for k in K_TABLE:
            pair_accs = multi[method][k]

            if method.startswith("sras"):
                complexity = "dataset-level O(K^3)"
            else:
                complexity = "pointwise O(n_pw K^3)"

            note = "Comparable subset row"
            if method == "msa":
                note = "Down-projected exact spectral-ratio baseline on subset"
            elif method == "pwairm":
                note = "Pointwise AIRM baseline on subset"

            fam_summary_rows.append(
                make_summary_row(
                    method=method,
                    family=family_name,
                    regime="subset_256",
                    K=k,
                    n_images=N_POINTWISE,
                    pair_accs=pair_accs,
                    comparison_cost=complexity,
                    note=note,
                )
            )
            fam_pair_rows.extend(make_pair_rows(method, family_name, "subset_256", k, pair_accs))

    del subset_ps_lookup
    del subset_mean_lookup
    gc.collect()

    return fam_summary_rows, fam_pair_rows

for family_name, fam in families.items():
    rows_s, rows_p = compute_family_rows(
        family_name=family_name,
        P_ref=fam["P_ref"],
        P_pointwise=fam["P_pointwise"],
    )
    summary_rows.extend(rows_s)
    pair_rows.extend(rows_p)

summary_df = pd.DataFrame(summary_rows)
pair_df = pd.DataFrame(pair_rows)

display(summary_df.sort_values(["family", "regime", "method", "K"]))

In [ ]:
# @title 7. Save summary CSVs and build table-ready files

summary_df = pd.DataFrame(summary_rows)
pair_df = pd.DataFrame(pair_rows)

summary_df.to_csv(DRIVE_OUTDIR / "table_summary_long.csv", index=False)
pair_df.to_csv(DRIVE_OUTDIR / "table_pair_accuracies_long.csv", index=False)

def fmt_pm(mean_val, sem_val, nd=1, nd_sem=2):
    return f"{mean_val:.{nd}f} ± {sem_val:.{nd_sem}f}"

summary_df["formatted"] = summary_df.apply(
    lambda r: fmt_pm(r["pair_accuracy_mean_pct"], r["pair_accuracy_sem_pct"]),
    axis=1,
)

# Activation rows for the activation block
activation_block = summary_df[
    summary_df["family"] == "activation"
].copy()
activation_block.to_csv(DRIVE_OUTDIR / "table_activation_block.csv", index=False)

# Local-geometry rows for the revised compact table
local_block = summary_df[
    (summary_df["family"] != "activation") &
    (summary_df["method"].isin(["sras_full", "sras_shape", "pwairm", "msa"]))
].copy()
local_block.to_csv(DRIVE_OUTDIR / "table_local_geometry_block.csv", index=False)

# Compact wide version for the specific table rewrite
wide_rows = []
for method in ["sras_full", "pwairm", "msa"]:
    for k in K_TABLE:
        row = {"method": method, "K": k}
        for fam in families.keys():
            hit = local_block[
                (local_block["method"] == method) &
                (local_block["family"] == fam) &
                (local_block["regime"] == "subset_256") &
                (local_block["K"].astype(str) == str(k))
            ]
            if len(hit) == 1:
                row[f"{fam}_formatted"] = hit["formatted"].iloc[0]
                row[f"{fam}_mean_pct"] = hit["pair_accuracy_mean_pct"].iloc[0]
                row[f"{fam}_sem_pct"] = hit["pair_accuracy_sem_pct"].iloc[0]
        wide_rows.append(row)

# Higher-K S-RAS references
for fam in families.keys():
    hit = local_block[
        (local_block["method"] == "sras_full") &
        (local_block["family"] == fam) &
        (local_block["regime"] == "full_ref_2000") &
        (local_block["K"].astype(str) == str(K_REF))
    ]
    if len(hit) == 1:
        wide_rows.append({
            "method": "sras_full_ref",
            "K": K_REF,
            "family": fam,
            "formatted": hit["formatted"].iloc[0],
            "mean_pct": hit["pair_accuracy_mean_pct"].iloc[0],
            "sem_pct": hit["pair_accuracy_sem_pct"].iloc[0],
        })

wide_df = pd.DataFrame(wide_rows)
wide_df.to_csv(DRIVE_OUTDIR / "table_local_geometry_wide.csv", index=False)

print("Saved:")
print(" ", DRIVE_OUTDIR / "table_summary_long.csv")
print(" ", DRIVE_OUTDIR / "table_pair_accuracies_long.csv")
print(" ", DRIVE_OUTDIR / "table_activation_block.csv")
print(" ", DRIVE_OUTDIR / "table_local_geometry_block.csv")
print(" ", DRIVE_OUTDIR / "table_local_geometry_wide.csv")

print("\nActivation block:")
display(
    activation_block[[
        "method", "regime", "n_images",
        "pair_accuracy_mean_pct", "pair_accuracy_sem_pct",
        "comparison_cost", "note"
    ]].sort_values(["method", "regime"])
)

print("\nLocal-geometry block:")
display(
    local_block[[
        "method", "family", "regime", "K", "n_images",
        "pair_accuracy_mean_pct", "pair_accuracy_sem_pct",
        "comparison_cost", "note"
    ]].sort_values(["method", "family", "regime", "K"])
)

In [ ]:
# @title 8. Print exact values for LaTeX table filling

summary_df = pd.read_csv(DRIVE_OUTDIR / "table_summary_long.csv")

def get_val(method, family, regime, K=None):
    df = summary_df[
        (summary_df["method"] == method) &
        (summary_df["family"] == family) &
        (summary_df["regime"] == regime)
    ].copy()

    if K is None:
        df = df[df["K"].astype(str).isin(["", "nan"])]
    else:
        df = df[df["K"].astype(str) == str(K)]

    if len(df) != 1:
        return None

    r = df.iloc[0]
    return f"{r['pair_accuracy_mean_pct']:.1f} ± {r['pair_accuracy_sem_pct']:.2f}"

print("=== Activation rows ===")
print("CKA full:", get_val("cka_linear", "activation", "full_2000"))
print("CKA subset:", get_val("cka_linear", "activation", "subset_256"))
print("Procrustes subset:", get_val("procrustes", "activation", "subset_256"))
if RUN_PROCRUSTES_FULL:
    print("Procrustes full:", get_val("procrustes", "activation", "full_2000"))

for fam in families.keys():
    print(f"\n=== {fam} higher-K references ===")
    print("S-RAS full @ 768:", get_val("sras_full", fam, "full_ref_2000", 768))
    print("S-RAS shape @ 768:", get_val("sras_shape", fam, "full_ref_2000", 768))

    print(f"\n=== {fam} comparable subset rows ===")
    for method in ["sras_full", "sras_shape", "pwairm", "msa"]:
        vals = []
        for k in K_TABLE:
            vals.append(f"K={k}: {get_val(method, fam, 'subset_256', k)}")
        print(method, " | ".join(vals))

In [ ]:
# @title Full activation-alignment block on 2000 images: linear CKA, RBF CKA, CCA, Procrustes, SVCCA

# This cell assumes previous notebook variables exist:
#   loaded_models, LAYER_NAMES, X_full_sharded, N_FULL, ACT_CACHE_CHUNK, DRIVE_OUTDIR
# It reuses act_cache_full if present; otherwise it builds it.

import numpy as np
import pandas as pd
import scipy.stats as stats
from tqdm.notebook import tqdm
import gc

# -------------------------
# Helpers
# -------------------------

def _center_rows(X):
    return X - X.mean(axis=0, keepdims=True)

def _build_full_activation_cache_if_needed():
    global act_cache_full
    cache_ok = False
    if "act_cache_full" in globals():
        try:
            sample_entry = act_cache_full[0][LAYER_NAMES[0]]
            cache_ok = isinstance(sample_entry, dict) and ("Xc" in sample_entry)
        except Exception:
            cache_ok = False

    if cache_ok:
        print("Reusing existing act_cache_full.")
        return

    print("Building act_cache_full from X_full_sharded...")
    act_cache_full = {}
    for i in tqdm(range(len(loaded_models)), desc="Activation cache full (2000)"):
        act_cache_full[i] = {}
        for layer in LAYER_NAMES:
            X = get_activation_matrix_cpu_chunked(
                loaded_models[i],
                layer,
                X_full_sharded,
                chunk_size=ACT_CACHE_CHUNK,
            )
            Xc = _center_rows(X).astype(np.float32)
            act_cache_full[i][layer] = {"Xc": Xc}
        gc.collect()

def _get_linear_gram(Xc):
    return (Xc @ Xc.T).astype(np.float32)

def _double_center_gram(K):
    n = K.shape[0]
    one_n = np.ones((n, n), dtype=K.dtype) / n
    return K - one_n @ K - K @ one_n + one_n @ K @ one_n

def _pairwise_sq_dists_from_rows(X):
    sq = np.sum(X * X, axis=1, keepdims=True)
    D = sq + sq.T - 2.0 * (X @ X.T)
    return np.maximum(D, 0.0)

def _rbf_gram_median_heuristic(Xc):
    D2 = _pairwise_sq_dists_from_rows(Xc.astype(np.float32))
    tri = D2[np.triu_indices(D2.shape[0], k=1)]
    tri = tri[tri > 0]
    if tri.size == 0:
        sigma2 = 1.0
    else:
        sigma2 = float(np.median(tri))
        sigma2 = max(sigma2, 1e-12)
    K = np.exp(-D2 / (2.0 * sigma2)).astype(np.float32)
    return _double_center_gram(K).astype(np.float32)

def _cka_from_grams(K, L):
    num = np.trace(K @ L)
    den = np.linalg.norm(K) * np.linalg.norm(L) + 1e-12
    return float(num / den)

def _left_svd_factors_from_Xc(Xc, floor=1e-12):
    """
    Thin left singular factors from sample-side Gram XX^T.
    Safe when feature dimensions differ across layers.
    """
    G = (Xc @ Xc.T).astype(np.float64)
    G = 0.5 * (G + G.T)
    evals, U = np.linalg.eigh(G)
    keep = evals > floor
    evals = evals[keep]
    U = U[:, keep]
    s = np.sqrt(np.maximum(evals, floor))
    order = np.argsort(s)[::-1]
    s = s[order].astype(np.float32)
    U = U[:, order].astype(np.float32)
    return U, s

def _ensure_svd_cache(entry):
    if ("U" not in entry) or ("s" not in entry):
        U, s = _left_svd_factors_from_Xc(entry["Xc"])
        entry["U"] = U
        entry["s"] = s

def _cca_corrs_from_entries(entry_x, entry_y):
    """
    Standard CCA canonical correlations between two centered representations.
    Using sample-side orthonormal bases.
    """
    _ensure_svd_cache(entry_x)
    _ensure_svd_cache(entry_y)
    Ux, Uy = entry_x["U"], entry_y["U"]
    s = np.linalg.svd(Ux.T @ Uy, compute_uv=False, full_matrices=False)
    s = np.clip(s.astype(np.float32), 0.0, 1.0)
    return s

def _cca_r2_from_entries(entry_x, entry_y):
    corrs = _cca_corrs_from_entries(entry_x, entry_y)
    return float(np.mean(corrs ** 2))

def _procrustes_similarity_from_entries(entry_x, entry_y):
    """
    Exact orthogonal Procrustes similarity:
        ||X^T Y||_* / (||X||_F ||Y||_F)
    computed through thin sample-side SVD factors.
    """
    _ensure_svd_cache(entry_x)
    _ensure_svd_cache(entry_y)
    Ux, sx = entry_x["U"], entry_x["s"]
    Uy, sy = entry_y["U"], entry_y["s"]

    core = (sx[:, None] * (Ux.T @ Uy).astype(np.float32)) * sy[None, :]
    nuc = np.linalg.svd(core, compute_uv=False, full_matrices=False).sum()
    denom = (np.linalg.norm(sx) * np.linalg.norm(sy)) + 1e-12
    return float(nuc / denom)

def _svcca_mean_corr_from_entries(entry_x, entry_y, var_keep=0.99):
    """
    SVCCA:
      1) SVD-compress each representation to retain var_keep fraction of variance
      2) run CCA on the retained subspaces
    Since CCA is invariant to per-component scaling, orthonormal retained U bases suffice.
    """
    _ensure_svd_cache(entry_x)
    _ensure_svd_cache(entry_y)

    sx = entry_x["s"].astype(np.float64)
    sy = entry_y["s"].astype(np.float64)
    Ux = entry_x["U"]
    Uy = entry_y["U"]

    vx = np.cumsum(sx ** 2) / np.sum(sx ** 2)
    vy = np.cumsum(sy ** 2) / np.sum(sy ** 2)

    rx = int(np.searchsorted(vx, var_keep) + 1)
    ry = int(np.searchsorted(vy, var_keep) + 1)

    Uxr = Ux[:, :rx]
    Uyr = Uy[:, :ry]

    corrs = np.linalg.svd(Uxr.T @ Uyr, compute_uv=False, full_matrices=False)
    corrs = np.clip(corrs.astype(np.float32), 0.0, 1.0)
    return float(np.mean(corrs))

def _ensure_linear_and_rbf_grams(entry, do_rbf=True):
    if "K_linear" not in entry:
        entry["K_linear"] = _get_linear_gram(entry["Xc"])
    if do_rbf and ("K_rbf" not in entry):
        entry["K_rbf"] = _rbf_gram_median_heuristic(entry["Xc"])

def _pair_accuracy_from_sim_map(sim_map):
    n = sim_map.shape[0]
    hits = 0
    for i in range(n):
        hits += int(np.argmax(sim_map[i, :]) == i)
    for j in range(n):
        hits += int(np.argmax(sim_map[:, j]) == j)
    return 100.0 * hits / (2 * n)

def _evaluate_similarity_metric_pair_accs(item_lookup, metric_fn, desc=None):
    n_layers = len(LAYER_NAMES)
    pair_accs = []
    iterator = tqdm(PAIR_LIST, desc=desc, leave=False) if desc is not None else PAIR_LIST

    for idx1, idx2 in iterator:
        sim_map = np.empty((n_layers, n_layers), dtype=np.float32)
        for i, l1 in enumerate(LAYER_NAMES):
            x1 = item_lookup[idx1][l1]
            for j, l2 in enumerate(LAYER_NAMES):
                x2 = item_lookup[idx2][l2]
                sim_map[i, j] = metric_fn(x1, x2)
        pair_accs.append(_pair_accuracy_from_sim_map(sim_map))

    return np.array(pair_accs, dtype=np.float32)

def _make_summary_row(method, regime, n_images, pair_accs, comparison_cost, note=""):
    pair_accs = np.asarray(pair_accs, dtype=np.float64)
    return {
        "method": method,
        "family": "activation",
        "regime": regime,
        "K": "",
        "n_images": int(n_images),
        "n_pairs": int(len(pair_accs)),
        "pair_accuracy_mean_pct": float(pair_accs.mean()),
        "pair_accuracy_sem_pct": float(stats.sem(pair_accs)) if len(pair_accs) > 1 else 0.0,
        "overall_accuracy_pct": float(pair_accs.mean()),
        "comparison_cost": comparison_cost,
        "note": note,
    }

def _make_pair_rows(method, regime, pair_accs):
    rows = []
    for (m1, m2), acc in zip(PAIR_LIST, pair_accs):
        rows.append({
            "method": method,
            "family": "activation",
            "regime": regime,
            "K": "",
            "model_i": int(m1),
            "model_j": int(m2),
            "pair_accuracy_pct": float(acc),
        })
    return rows

# -------------------------
# Build / reuse full cache
# -------------------------

_build_full_activation_cache_if_needed()

# Add SVD factors lazily once for all full entries
print("Preparing sample-side SVD factors on full set...")
for i in tqdm(range(len(loaded_models)), desc="Full SVD factors"):
    for layer in LAYER_NAMES:
        _ensure_svd_cache(act_cache_full[i][layer])
    gc.collect()

# Add linear + RBF grams
print("Preparing linear and RBF Grams on full set...")
for i in tqdm(range(len(loaded_models)), desc="Full linear/RBF Grams"):
    for layer in LAYER_NAMES:
        _ensure_linear_and_rbf_grams(act_cache_full[i][layer], do_rbf=True)
    gc.collect()

PAIR_LIST = list(itertools.combinations(range(len(loaded_models)), 2))

# -------------------------
# Evaluate all five methods
# -------------------------

full_activation_rows = []
full_activation_pair_rows = []

cka_linear_full_pair_accs = _evaluate_similarity_metric_pair_accs(
    act_cache_full,
    metric_fn=lambda ex, ey: _cka_from_grams(ex["K_linear"], ey["K_linear"]),
    desc="Linear CKA full (2000)",
)
full_activation_rows.append(
    _make_summary_row(
        method="cka_linear",
        regime="full_2000",
        n_images=N_FULL,
        pair_accs=cka_linear_full_pair_accs,
        comparison_cost="activation Gram",
        note="Full Tiny10 benchmark",
    )
)
full_activation_pair_rows.extend(_make_pair_rows("cka_linear", "full_2000", cka_linear_full_pair_accs))

cka_rbf_full_pair_accs = _evaluate_similarity_metric_pair_accs(
    act_cache_full,
    metric_fn=lambda ex, ey: _cka_from_grams(ex["K_rbf"], ey["K_rbf"]),
    desc="RBF CKA full (2000)",
)
full_activation_rows.append(
    _make_summary_row(
        method="cka_rbf",
        regime="full_2000",
        n_images=N_FULL,
        pair_accs=cka_rbf_full_pair_accs,
        comparison_cost="RBF Gram",
        note="Median-heuristic RBF CKA on full 2000-image set",
    )
)
full_activation_pair_rows.extend(_make_pair_rows("cka_rbf", "full_2000", cka_rbf_full_pair_accs))

cca_full_pair_accs = _evaluate_similarity_metric_pair_accs(
    act_cache_full,
    metric_fn=_cca_r2_from_entries,
    desc="CCA (R^2) full (2000)",
)
full_activation_rows.append(
    _make_summary_row(
        method="cca_r2",
        regime="full_2000",
        n_images=N_FULL,
        pair_accs=cca_full_pair_accs,
        comparison_cost="sample-side SVD + CCA",
        note="Mean squared canonical correlation on full 2000-image set",
    )
)
full_activation_pair_rows.extend(_make_pair_rows("cca_r2", "full_2000", cca_full_pair_accs))

procrustes_full_pair_accs = _evaluate_similarity_metric_pair_accs(
    act_cache_full,
    metric_fn=_procrustes_similarity_from_entries,
    desc="Procrustes full (2000)",
)
full_activation_rows.append(
    _make_summary_row(
        method="procrustes",
        regime="full_2000",
        n_images=N_FULL,
        pair_accs=procrustes_full_pair_accs,
        comparison_cost="sample-side SVD",
        note="Exact orthogonal Procrustes on full 2000-image set",
    )
)
full_activation_pair_rows.extend(_make_pair_rows("procrustes", "full_2000", procrustes_full_pair_accs))

svcca_full_pair_accs = _evaluate_similarity_metric_pair_accs(
    act_cache_full,
    metric_fn=lambda ex, ey: _svcca_mean_corr_from_entries(ex, ey, var_keep=0.99),
    desc="SVCCA full (2000)",
)
full_activation_rows.append(
    _make_summary_row(
        method="svcca_bar_rho",
        regime="full_2000",
        n_images=N_FULL,
        pair_accs=svcca_full_pair_accs,
        comparison_cost="sample-side SVD + CCA",
        note="SVCCA with 99% variance retention on full 2000-image set",
    )
)
full_activation_pair_rows.extend(_make_pair_rows("svcca_bar_rho", "full_2000", svcca_full_pair_accs))

full_activation_summary_df = pd.DataFrame(full_activation_rows)
full_activation_pair_df = pd.DataFrame(full_activation_pair_rows)

display(full_activation_summary_df.sort_values("method"))

# Save
save_dir = DRIVE_OUTDIR if "DRIVE_OUTDIR" in globals() else Path("/content")
full_activation_summary_df.to_csv(save_dir / "full_activation_alignment_summary_2000.csv", index=False)
full_activation_pair_df.to_csv(save_dir / "full_activation_alignment_pair_accuracies_2000.csv", index=False)

print("\nSaved:")
print(" ", save_dir / "full_activation_alignment_summary_2000.csv")
print(" ", save_dir / "full_activation_alignment_pair_accuracies_2000.csv")